In [2]:
import os
from utils_extraction import extract_body, tokenize, clean_tokens, decode, is_auto_label_tag
from utils_extraction import chunk_tokens, flatten_token_chunks
from utils_extraction import extract_few_shot_examples
from utils_extraction import select_few_shot
from utils_extraction import merge_tokens_general, merge_tokens_with_auto_labels, add_attributes_to_auto_labels, compare_html_allow_auto_labels, correct_tokens_brackets, check_tokens_brackets
from models import GPTAssistant
from utils_extraction import process_chunks
from utils_extraction.html_utils import clean_html_formatting
import json

project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"


In [3]:
# ---------- Define Hyperparameters ----------
min_tokens = 500
fs_min_tokens = 100
fs_mode = "random"  # "random" or "selected"
model_name = "gpt-5.2"

n_few_shot = 30  # Number of few-shot examples to use

prompt_version = "2"
cot = False
if cot :
    prompt_path = fr"{project_root}\llm_based_annotation\utils_extraction\prompts\simplified_parent_extraction_cot v{prompt_version}.txt"
else :
    prompt_path = fr"{project_root}\llm_based_annotation\utils_extraction\prompts\simplified_parent_extraction v{prompt_version}.txt"


#### Define the text to process, and where to save it. Define the text for few shot

In [4]:
# File paths
filename = '2001CanLII21117' #"2021QCCA1675" #"1989CanLII1415ONCA" #"2021QCCA1675" #"1997CanLII16226_ONCA"
round = "ronde_2"
anno = "llm"
version = "v1.0"

try:
    ext = "html"
    html_path = fr"{project_root}\data\Document_Échantillon_Initial\{round}\plain_html_arbre_balise\{filename}.{ext}"
    with open(html_path, 'r', encoding='utf-8') as file:
        html_content = file.read()
    

except Exception as e:
    ext = "htm"
    html_path = fr"{project_root}\data\Document_Échantillon_Initial\{round}\plain_html_arbre_balise\{filename}.{ext}"
    # Read HTML file
    with open(html_path, 'r', encoding='utf-8') as file:
        html_content = file.read()

print(f"   ✓ HTML file loaded: {html_path}")
if fs_mode == "selected" :
    output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}\v_prompt_{prompt_version}_{min_tokens}_{fs_mode}_{n_few_shot}_gpt5.2_chunk2_subdef2"
else :
    output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}\v_prompt_{prompt_version}_{min_tokens}_{fs_min_tokens}_{n_few_shot}_gpt5.2_chunk2_subdef2"
os.makedirs(output_dir, exist_ok=True)




   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Document_Échantillon_Initial\ronde_2\plain_html_arbre_balise\2001CanLII21117.html


### Process The HTML Content

In [5]:

# ---------- Extract body content ----------
body_content = extract_body(html_content)


# ---------- Tokenize body content ----------
tokens = tokenize(body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
#token_chunks = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)


In [6]:
for tok1, tok2 in zip(normalized_cleaned_tokens, tokenize(decode(normalized_cleaned_tokens))):
    if tok1 != tok2:
        print(tok1, tok2)
        break

In [7]:
import spacy
import re

nlp = spacy.load("en_core_web_trf")

doc = nlp(decode(normalized_cleaned_tokens))
initial_sentences = [sent.text for sent in doc.sents]



c:\Users\zakga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
initial_sentences_token = [tokenize(sent) for sent in initial_sentences]
flat_initial_sentences = flatten_token_chunks(initial_sentences_token, separator="<sep>")

   ✓ Flattened 801 chunks into 37638 tokens


In [9]:
def merge_tokens_general(original_tokens: list[str], 
                        derived_tokens: list[str], 
                        is_protected_func,
                        log: bool = False) -> list[str]:
    """
    GENERALIZED VERSION: Merge original tokens with derived tokens.
    
    Goal: Produce the original text with protected tokens (e.g., <sep>, <auto_label>) 
    inserted from the derived version.
    
    Algorithm:
    - If tokens are equivalent (same or both whitespace): take original token, advance both indices
    - If tokens differ:
      - If derived token is protected: it's an insertion, take it and advance idx2 only
      - Otherwise: try to merge consecutive original tokens to match derived token
      - If no merge possible: take original token and advance idx1 only
    
    This assumes derived is mostly a superset of original (original + protected tokens).
    
    Args:
        original_tokens: Original token list (without protected tokens)
        derived_tokens: Derived token list (with protected tokens inserted)
        is_protected_func: Function that takes a token and returns True if it's protected
        log: Print debug information
    
    Returns:
        Merged token list with original tokens + protected tokens from derived
    
    Example:
        original = ['Act', ',', 'section', '5']
        derived = ['Act', '<sep>', ',', 'section', '<sep>', '5']
        is_protected = lambda t: t == '<sep>'
        -> ['Act', '<sep>', ',', 'section', '<sep>', '5']
    """
    n1 = len(original_tokens)
    n2 = len(derived_tokens)
    result = []
    idx1 = 0
    idx2 = 0
    
    def tokens_equivalent(tok1: str, tok2: str) -> bool:
        """Check if two tokens are equivalent (exact match or both whitespace)."""
        if tok1 == tok2:
            return True
        # Both are pure whitespace
        if not tok1.strip() and not tok2.strip():
            return True
        return False
    
    def try_merge_original_to_match_derived(start_idx: int, target: str) -> int:
        """
        Try to merge consecutive original tokens to match the derived token.
        
        Example: If original=['générale', '"'] and derived='généraleˮ',
        this will detect that original[0] + original[1] matches derived.
        
        Args:
            start_idx: Starting index in original_tokens
            target: The derived token to match
            
        Returns:
            Number of original tokens that combine to match target (0 if no match)
        """
        if start_idx >= n1:
            return 0
        
        accumulated = ""
        # Look ahead up to 10 tokens to find a match
        for i in range(start_idx, min(start_idx + 10, n1)):
            accumulated += original_tokens[i]
            if accumulated == target:
                return i - start_idx + 1
        return 0
    
    while idx1 < n1 and idx2 < n2:
        t1 = original_tokens[idx1]
        t2 = derived_tokens[idx2]
        
        if tokens_equivalent(t1, t2):
            # Tokens match: keep original and advance both
            result.append(t1)
            if log:
                print(f"Match: '{t1}' == '{t2}' -> '{t1}'")
            idx1 += 1
            idx2 += 1
        else:
            # Tokens differ
            if is_protected_func(t2):
                # t2 is a protected token (insertion in derived version)
                result.append(t2)
                if log:
                    print(f"Protected: '{t1}' vs '{t2}' -> '{t2}'")
                idx2 += 1
            else:
                # Neither matches nor is protected
                # Try to merge original tokens to match derived token
                merge_count = try_merge_original_to_match_derived(idx1, t2)
                
                if merge_count > 0:
                    # Found a match by merging multiple original tokens
                    for i in range(merge_count):
                        result.append(original_tokens[idx1 + i])
                    if log:
                        merged_tokens = original_tokens[idx1:idx1+merge_count]
                        print(f"Merged {merge_count} tokens: {merged_tokens} -> '{t2}'")
                    idx1 += merge_count
                    idx2 += 1
                else:
                    # No merge possible: keep original token
                    result.append(t1)
                    if log:
                        print(f"Diff: '{t1}' vs '{t2}' -> '{t1}'")
                    idx1 += 1
    
    # Append remaining tokens from original (if any)
    if idx1 < n1:
        result.extend(original_tokens[idx1:])
    
    # Append remaining tokens from derived (if any, likely protected tokens)
    if idx2 < n2:
        result.extend(derived_tokens[idx2:])
    
    if log:
        print(f"   ✓ Merged {n1} original + {n2} derived → {len(result)} tokens")
        print(f"   ✓ Added {len(result) - n1} protected tokens")
    
    return result

In [10]:
is_sep_tag = lambda token: token == '<sep>'

# Example: merge normalized_cleaned_tokens (original, no <sep>) 
# with flat_token_sentence_chunks (derived, with <sep>)

print(f"Original tokens: {len(normalized_cleaned_tokens)} (no <sep>)")
print(f"Derived tokens: {len(flat_initial_sentences)} (with <sep>)")

# This assumes flat_token_sentence_chunks is normalized_cleaned_tokens + <sep> insertions
corrected_initial_sentences = merge_tokens_general(
    original_tokens=normalized_cleaned_tokens,
    derived_tokens=flat_initial_sentences,
    is_protected_func=is_sep_tag,
    log=False
)

print(f"\nResult: {len(corrected_initial_sentences)} tokens")
print(f"Number of <sep> tags: {corrected_initial_sentences.count('<sep>')}")

Original tokens: 37462 (no <sep>)
Derived tokens: 37638 (with <sep>)

Result: 38262 tokens
Number of <sep> tags: 800


In [11]:
def calculate_combined_density(sentence):
    """Calculate combined period and number density for a sentence."""
    if len(sentence) < 10:
        return 0
    
    num_periods = sentence.count('.')
    num_digits = sum(c.isdigit() for c in sentence)
    
    period_density = (num_periods / len(sentence)) * 100
    digit_density = (num_digits / len(sentence)) * 100
    
    return period_density + digit_density

def detect_citation_sections_sequential(sentences, threshold=25, consecutive_gap=3):
    """
    Detect citation sections using sequential analysis.
    
    Algorithm:
    1. Go through sentences one by one
    2. When density > threshold, start a citation section
    3. Continue until we find 'consecutive_gap' sentences below threshold
    4. End the citation section 'consecutive_gap' sentences before
    5. STOP - all remaining sentences are NOT citations
    
    Args:
        sentences: List of sentences
        threshold: Combined density threshold (%) to consider citation
        consecutive_gap: Number of consecutive non-citation sentences to end section
    
    Returns:
        List of booleans indicating if each sentence is in a citation section
    """
    is_citation = [False] * len(sentences)
    in_citation_section = False
    citation_start = None
    below_threshold_count = 0
    citation_section_ended = False  # Track if we've already found and ended a citation section
    
    for i, sent in enumerate(sentences):
        # If citation section has already ended, all remaining sentences are NOT citations
        if citation_section_ended:
            break
        
        density = calculate_combined_density(sent)
        
        if density > threshold:
            # This is a citation sentence
            if not in_citation_section:
                # Start new citation section
                in_citation_section = True
                citation_start = i
            # Reset the gap counter
            below_threshold_count = 0
            is_citation[i] = True
            
        else:
            # Below threshold
            if in_citation_section:
                # We're in a citation section, count consecutive non-citations
                below_threshold_count += 1
                
                if below_threshold_count >= consecutive_gap:
                    # End citation section: go back 'consecutive_gap' sentences
                    citation_end = i - consecutive_gap
                    # Mark all sentences in this section
                    for j in range(citation_start, citation_end + 1):
                        is_citation[j] = True
                    # STOP HERE - citation section has ended
                    citation_section_ended = True
                    in_citation_section = False
    
    # Handle case where document ends while in citation section
    if in_citation_section and citation_start is not None:
        citation_end = len(sentences) - 1 - below_threshold_count
        for j in range(citation_start, citation_end + 1):
            is_citation[j] = True
    
    return is_citation

def merge_sentences_with_heuristics_tokens(tokens, citation_threshold=25, min_token=500):
    """
    Merge sentences based on boundary heuristics, working with tokens.
    
    Input : Flat list of tokens with <sep> as sentence boundaries.
    Output: Flat list of tokens with selective <sep> removal based on citation detection.

    Rules:
    - Normal section: Keep <sep> only if sentence ends with '.'
    - Citation section: Keep <sep> only if sentence ends with ';'
    - Otherwise: Merge sentences together (remove <sep>)
    
    Args:
        tokens: Flat List of token containing <sep> as sentence boundaries
        citation_threshold: Combined density threshold (%) for citation detection

    Returns:
        Flat list of tokens with selective <sep> removal based on citation detection
    """
    if not tokens:
        return []
    
    # STEP 1: Split into sentences based on <sep>
    sentences_tokens = []
    current_sentence = []
    for token in tokens:
        if token == '<sep>':
            if current_sentence:
                sentences_tokens.append(current_sentence)
                current_sentence = []
        else:
            current_sentence.append(token)
    
    if current_sentence:
        sentences_tokens.append(current_sentence)

    # STEP 2: Convert to text for citation detection
    sentences_text = [decode(sent) for sent in sentences_tokens]

    # STEP 3: Detect citations section
    is_citation = detect_citation_sections_sequential(sentences_text, threshold=citation_threshold)

    # STEP 4: Reconstruct flat token list with selective <sep> removal
    result = []
    current_chunk_size = 0  # Track tokens since last <sep>
    
    for i, sent_tokens in enumerate(sentences_tokens):
        result.extend(sent_tokens)
        current_chunk_size += len(sent_tokens)

        # Decide wheter to add <sep> after this sentence
        if i < len(sentences_tokens) - 1:  # Not the last sentence
            should_keep_sep = False

            # Get the last token of current sentence
            last_token = sent_tokens[-1] if sent_tokens else ''

            # Only consider keeping <sep> if we have at least min_token accumulated
            if current_chunk_size >= min_token:
                if is_citation[i]:
                    # Citation section: keep <sep> only if last token is with ';'
                    if last_token == ";":
                        should_keep_sep = True
                else:
                    # Normal section: keep <sep> only if last token is with '.'
                    if last_token == ".":
                        should_keep_sep = True
            
            if should_keep_sep:
                result.append('<sep>')
                current_chunk_size = 0  # Reset chunk size after keeping <sep>
    
    return result

In [12]:
# Apply heuristic-based merging with sequential citation detection
CITATION_THRESHOLD = 25  # Combined density threshold (%) - adjust based on the graphs above

flat_token_sentence_chunks = merge_sentences_with_heuristics_tokens(corrected_initial_sentences, citation_threshold=CITATION_THRESHOLD)
print(f"Initial sentences: {len(corrected_initial_sentences)}, After merging: {len(flat_token_sentence_chunks)}")
print(f"Using citation threshold: {CITATION_THRESHOLD}% (combined period + number density)")
print(f"Gap tolerance: 3 consecutive sentences below threshold to end citation section")
print(f"Note: Only the FIRST citation section is detected; all subsequent sentences are not citations")

Initial sentences: 38262, After merging: 37527
Using citation threshold: 25% (combined period + number density)
Gap tolerance: 3 consecutive sentences below threshold to end citation section
Note: Only the FIRST citation section is detected; all subsequent sentences are not citations


In [13]:
token_chunks =  []
current_chunk = []

for token in flat_token_sentence_chunks:

    if token != "<sep>":
        current_chunk.append(token)
    else:
        token_chunks.append(current_chunk)
        current_chunk =  []
token_chunks.append(current_chunk)

In [14]:
# Verify: merged should equal normalized_cleaned_tokens if ignoring <sep> tag

if flatten_token_chunks(token_chunks) == normalized_cleaned_tokens:
    print("✓ Perfect match! Derived was indeed original + <sep> insertions")
else:
    print("⚠ Some differences exist beyond <sep> insertions")
    # Show first difference
    for i, (m, d) in enumerate(zip(flat_token_sentence_chunks, flat_token_sentence_chunks)):
        if m != d:
            print(f"  First diff at index {i}: merged='{m}' vs derived='{d}'")
            break
    assert "Difference detected"

   ✓ Flattened 66 chunks into 37462 tokens
✓ Perfect match! Derived was indeed original + <sep> insertions


### Few Shot selection

In [15]:
# Define label_config 
label_config = {
    "keep_attributes": ["labelname"],  # extraction only, no disambiguation
    "switch_type": True,  # manual_label -> auto_label
    "use_simplified": True,  # <auto_label labelname="title"> -> <title>
    "keep_labels": ["decision", "legislation", "secondary sources"]
}

In [16]:
# ---------- Extract body content ----------
if fs_mode == "random" :
    fs_filename = "2019SCC65_annotated_EG_tech_corrected"
    fs_anno = "EG"
    fs_version = "v1"
    fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}.html"
    # Read HTML file
    with open(fs_html_path, 'r', encoding='utf-8') as file:
        fs_html_content = file.read()
    print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")
    fs_body_content = extract_body(fs_html_content)


    # ---------- Tokenize body content ----------
    fs_tokens = tokenize(fs_body_content)

    # ---------- Clean tokens ----------
    normalized_cleaned_tokens = clean_tokens(html_tokens=fs_tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

    # ---------- Chunk tokens ----------
    token_chunk1, token_chunk2 = chunk_tokens(normalized_cleaned_tokens, min_tokens=fs_min_tokens, stop_bookmark_separation=True)


    label_config = {
    "keep_attributes":["labelname"], # extraction only, no disambiguation
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    "keep_labels":["decision", "legislation", "secondary sources"]
    }
    
    # ---------- Create few-shot examples ----------

    few_shot_examples = extract_few_shot_examples(token_chunk1, 
                                                label_config)



    selected_few_shot_examples = select_few_shot(examples=few_shot_examples, n=n_few_shot, 
                                                method="distributed", 
                                                list_of_labels=["decision", "legislation", "secondary sources"], 
                                                distribution=[0.3, 0.3, 0.3])
    print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")

    for examples in selected_few_shot_examples:
        print("==============================================================================")
        print("input: \n ", examples[0])
        print("------------------------------------------------------------------------------")
        print("output: \n ", examples[1])



   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\EG\2019SCC65_annotated_EG_tech_corrected.html
   ✓ Found bookmark separator at index 43561
   ✓ Splitting: 43561 tokens before, 102341 tokens after
   ✓ Chunked tokens into 278 chunks (>= 100 tokens each)
   ✓ Chunked tokens into 1000 chunks (>= 100 tokens each)
   ✓ Total chunks: 278 before + 1000 after = 1278
   ✓ Extracted 278 few-shot examples from chunks
   ⚠ Warning: Requested 9 examples with label 'secondary sources', but only 5 available
   ✓ Selected 30 few-shot examples for processing.
input: 
   these limited exceptions, offers a comprehensive approach to determining the applicable standard of review. As a result, it is no longer necessary for courts to engage in a “contextual inquiry” (CHRC, at paras. 45-47; see also Dunsmuir, at paras. 62-64; McLean, at para. 22) in
------------------------------------------------------------------------------
output:

In [ ]:
if fs_mode == "selected":
    
    # Load the selected few-shot examples JSON
    fs_json_path = fr"{project_root}\few_shot_selection_tool\second_selected\combined_v3_with_sources_fixed_spacing.json"
    with open(fs_json_path, 'r', encoding='utf-8') as file:
        fs_data = json.load(file)
    print(f"   ✓ Loaded {len(fs_data)} examples from: {fs_json_path}")
    
    # Get all unique source files and print them
    source_files = sorted(list(set([item.get('source_file', 'unknown') for item in fs_data])))
    print(f"\n   Source files in few-shot collection:")
    for sf in source_files:
        count = sum(1 for item in fs_data if item.get('source_file') == sf)
        print(f"      - {sf} ({count} examples)")
    
    # Apply filters:
    # 1. Mask filter: Exclude examples from the same document being annotated (avoid data leakage)
    # 2. Manual filter: Only keep examples where "selected" == true
    current_doc_base = filename
    
    filtered_examples = []
    excluded_same_doc = 0
    excluded_not_selected = 0
    
    for item in fs_data:
        source_file = item.get('source_file', '')
        
        # Filter 1: Check if the current document name appears in the source file (MASK FILTER)
        if current_doc_base in source_file:
            excluded_same_doc += 1
            continue
        
        # Filter 2: Only keep examples with "selected" == true (MANUAL FILTER)
        if not item.get('selected', False):
            excluded_not_selected += 1
            continue
        
        # Extract input/output from the example
        if 'example' in item and 'input' in item['example'] and 'output' in item['example']:
            filtered_examples.append({
                'input': item['example']['input'],
                'output': item['example']['output'],
                'source_file': source_file
            })
    
    print(f"\n   ✓ Filtering results:")
    print(f"      - Excluded (same document): {excluded_same_doc}")
    print(f"      - Excluded (not selected): {excluded_not_selected}")
    print(f"      - Retained: {len(filtered_examples)}")
    
    # Show source files of retained examples
    retained_sources = {}
    for ex in filtered_examples:
        sf = ex['source_file']
        retained_sources[sf] = retained_sources.get(sf, 0) + 1
    
    print(f"\n   ✓ Retained examples come from:")
    for sf, count in sorted(retained_sources.items()):
        print(f"      - {sf}: {count} examples")
    
    # Select the required number of examples
    if len(filtered_examples) > n_few_shot:
        selected_examples_dicts = filtered_examples[:n_few_shot]
    else:
        selected_examples_dicts = filtered_examples
    
    # Simplify outputs to parent-level extraction only (remove nested tags)
    def simplify_to_parent_level(html_text):
        """
        Simplify annotated text to parent-level only.
        Keep only: <legislation>, <decision>, <secondary sources>
        Remove nested tags: <title>, <citation>, <fragment>, <source>, <authors>
        
        Example: <legislation><title>Act</title></legislation> -> <legislation>Act</legislation>
        """
        parent_tags = ['legislation', 'decision', 'secondary sources']
        
        # Tokenize the HTML text
        tokens = tokenize(html_text)
        
        simplified_tokens = []
        depth = 0  # Track nesting depth
        parent_tag_stack = []  # Track which parent tags are open
        
        for token in tokens:
            # Check if it's an opening tag
            if token.startswith('<') and not token.startswith('</') and token.endswith('>'):
                tag_name = token[1:-1].split()[0]  # Get tag name without attributes
                
                if tag_name in parent_tags:
                    # This is a parent-level tag, keep it
                    simplified_tokens.append(token)
                    parent_tag_stack.append(tag_name)
                    depth += 1
                else:
                    # This is a nested tag, skip it but keep the content
                    depth += 1
            
            # Check if it's a closing tag
            elif token.startswith('</') and token.endswith('>'):
                tag_name = token[2:-1]
                depth -= 1
                
                if tag_name in parent_tags:
                    # This is a parent-level closing tag, keep it
                    simplified_tokens.append(token)
                    if parent_tag_stack and parent_tag_stack[-1] == tag_name:
                        parent_tag_stack.pop()
                # Nested closing tags are skipped
            
            else:
                # Regular text content, keep it
                simplified_tokens.append(token)
        
        # Decode back to string
        return decode(simplified_tokens)
    
    print(f"\n   ✓ Simplifying outputs to parent-level extraction...")
    simplified_examples = []
    for ex in selected_examples_dicts:
        simplified_output = simplify_to_parent_level(ex['output'])
        simplified_examples.append((ex['input'], simplified_output))
    
    # Convert to list of tuples (input, output)
    selected_few_shot_examples = simplified_examples
    
    print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")
    
    # Print first example as verification (before and after simplification)
    if selected_few_shot_examples and selected_examples_dicts:
        print("\n   First example preview:")
        print("   Input:", selected_few_shot_examples[0][0][:100], "...")
        print("   Output (original):", selected_examples_dicts[0]['output'][:150], "...")
        print("   Output (simplified):", selected_few_shot_examples[0][1][:150], "...")

   ✓ Loaded 270 examples from: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\few_shot_selection_tool\second_selected\combined_v3_with_sources_fixed_spacing.json

   Source files in few-shot collection:
      - few_shot_examples_1989CanLII1415ONCA_annotated_GL_tech.json (35 examples)
      - few_shot_examples_1997CanLII16226_ONCA_annotated_EG_tech.json (155 examples)
      - few_shot_examples_2019SCC65_annotated_EG_tech_corrected.json (59 examples)
      - few_shot_examples_2021QCCA1675_annotated_EG_tech.json (21 examples)

   ✓ Filtering results:
      - Excluded (same document): 155
      - Excluded (not selected): 94
      - Retained: 21

   ✓ Retained examples come from:
      - few_shot_examples_1989CanLII1415ONCA_annotated_GL_tech.json: 12 examples
      - few_shot_examples_2019SCC65_annotated_EG_tech_corrected.json: 4 examples
      - few_shot_examples_2021QCCA1675_annotated_EG_tech.json: 5 examples

   ✓ Simplifying outputs to parent-level extraction...
   ✓ S

In [17]:
selected_few_shot_examples

[(' these limited exceptions, offers a comprehensive approach to determining the applicable standard of review. As a result, it is no longer necessary for courts to engage in a “contextual inquiry” (CHRC, at paras.\xa045-47; see also Dunsmuir, at paras. 62-64; McLean, at para. 22) in',
  ' these limited exceptions, offers a comprehensive approach to determining the applicable standard of review. As a result, it is no longer necessary for courts to engage in a “contextual inquiry” (<decision>CHRC, at paras.\xa045-47</decision>; see also <decision>Dunsmuir, at paras. 62-64</decision>; <decision>McLean, at para. 22</decision>) in'),
 (' Cited\nArthurs, H.\xa0W. “Protection against Judicial Review” (1983), 43 R. du B. 277.\nBarak, Aharon. “Overruling Precedent” (1986), 21 Is.L.R. 269.\nBiddulph, Michelle. “Rethinking the Ramifications of Reasonableness Review: Stare Decisis and Reasonableness Review on Questions of Law” (2018), 56 Alta. L.R. 119.\nBingham, Tom. The Rule of Law. London: All

### Document processing

In [18]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name, temperature=1)

In [19]:
# ---------- Process chunks ----------
processed_chunks = process_chunks(
    model=model,
    token_chunks=token_chunks,
    process_prompt_path=prompt_path,
    label_config=label_config,
    few_shot_examples=selected_few_shot_examples,
    output_dir=output_dir,
    filename=filename,
    cot = cot,
)


   ✓ Processing 66 chunks with LLM...
   ✓ Using 30 few-shot examples


Processing chunks: 100%|██████████| 66/66 [06:31<00:00,  5.94s/it]

   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2001CanLII21117\v_prompt_2_500_100_30_gpt5.2_chunk2_subdef2\history_2001CanLII21117.json

   ✓ Processing completed:
      - Total chunks: 66
      - Successful: 66
      - Failed: 0
   ✓ Processed chunks saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2001CanLII21117\v_prompt_2_500_100_30_gpt5.2_chunk2_subdef2\processed_chunks_2001CanLII21117.json


In [20]:
#write the processed chuncks in a json file for later use in the annotation interface

import json 
with open(f"{output_dir}\\processed_chunks.json", "w") as f:
    json.dump(processed_chunks, f)

### Post Processing

In [5]:
# Read the processed_chuncks.json file to verify it was written correctly
import json
#with open(f"{output_dir}\\processed_chunks.json", "r") as f:
with open(f"C:\\Users\zakga\\OneDrive\\Documents\\code\\LeREaD_annotation_process\\data\Documents_Annotés\\llm\\p2_c500_fsselected-30_mgpt-5.2\\processed_chunks.json", "r") as f:
    processed_chunks = json.load(f)

<>:4: SyntaxWarning: invalid escape sequence '\z'
<>:4: SyntaxWarning: invalid escape sequence '\z'
C:\Users\zakga\AppData\Local\Temp\ipykernel_3936\3045996297.py:4: SyntaxWarning: invalid escape sequence '\z'
  with open(f"C:\\Users\zakga\\OneDrive\\Documents\\code\\LeREaD_annotation_process\\data\Documents_Annotés\\llm\\p2_c500_fsselected-30_mgpt-5.2\\processed_chunks.json", "r") as f:


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\zakga\\OneDrive\\Documents\\code\\LeREaD_annotation_process\\data\\Documents_Annotés\\llm\\p2_c500_fsselected-30_mgpt-5.2\\processed_chunks.json'

In [ ]:
# Processed_chunks is a list of lists of tokens, we need to flatten it to get a single list of tokens for the whole document
processed_tokens_flat = flatten_token_chunks(processed_chunks)


# Read in parallel the original tokens and the processed tokens. Always prefer the original tokens, but if there is an auto_label token in the processed tokens, 
# we want to keep it and merge it with the original tokens. 
# This way we can keep the original formatting and structure of the document while adding the auto_labels extracted by the model.
original_tokens = tokenize(html_content)
processed_html_content_tokens = merge_tokens_general(
    original_tokens=original_tokens,
    derived_tokens=processed_tokens_flat,
   is_protected_func=lambda tok: is_auto_label_tag(tok) != 0,
    log=False
)
#processed_html_content_tokens = merge_tokens_with_auto_labels(original_tokens, processed_tokens_flat)

# This merging process can sometimes create some formatting issues with brackets, we need to correct them to get a valid HTML structure.
processed_html_content_tokens_corrected = correct_tokens_brackets(processed_html_content_tokens)
assert check_tokens_brackets(processed_html_content_tokens_corrected), "The brackets in the merged tokens are not balanced. Please check the merging and bracket correction steps for errors."

processed_html_content_tokens_corrected = processed_html_content_tokens

# The correction of the brackets can sometimes create some redoundant or useless formatting  with the HTML, we need to clean it to compare it with the original.
processed_html = decode(processed_html_content_tokens_corrected)
processed_html_cleaned = clean_html_formatting(processed_html)
print(f"\nMerged HTML length: {len(processed_html_cleaned)}")

processed_html_cleaned = decode(processed_html_content_tokens_corrected)

# This step is just to ensure a good visualisation of HTMLLabelizer and to add the necessary attribute to stay consistent with the label scheme
processed_html_content = add_attributes_to_auto_labels(processed_html_cleaned)


# Last check of the final processed_html_content with the original HTML, ignoring the auto_label tags which are not present in the original HTML but only in the processed one.
comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)
assert comparison_result, "The processed HTML content does not match the original HTML content when ignoring auto_label tags. Please check the merging and post-processing steps for errors."


# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_llm_{version}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")

   ✓ Flattened 66 chunks into 38116 tokens

Merged HTML length: 259192
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)
   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2001CanLII21117\v_prompt_2_500_100_30_gpt5.2_chunk2_subdef2
